# Диплом: KAN и Hybrid KAN на ETT

Основной ноутбук для финальной экспериментальной главы ВКР.

Постановка: сравнить `KAN`, `Hybrid KAN`, `MLP`, `LSTM` и простые baseline-модели на benchmark-датасете `ETTh1` / `ETTm1` в задаче прогнозирования температуры трансформатора `OT`.

## 1. Почему ETT

- `ETT` является современным и широко используемым benchmark для time series forecasting.
- Датасет удобен для воспроизводимых экспериментов: есть стандартные split'ы, несколько горизонтов прогноза и компактный размер.
- Для диплома это сильнее, чем старые UCI-наборы, потому что `ETT` хорошо узнаётся в современной литературе по forecasting.
- Новая глава логично дополняет уже выполненные эксперименты на недвижимости, кредитных данных и климатических данных.

## 2. Импорты и настройки

In [ ]:
import io
import math
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests

import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

warnings.filterwarnings("ignore")
sns.set(style="whitegrid")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

## 3. Выбор датасета и загрузка

In [ ]:
DATASET_NAME = "ETTh1"  # варианты: ETTh1, ETTm1

DATA_URLS = {
    "ETTh1": "https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTh1.csv",
    "ETTm1": "https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTm1.csv",
}

TARGET_COL = "OT"
FEATURE_COLS = ["HUFL", "HULL", "MUFL", "MULL", "LUFL", "LULL", "OT"]

if DATASET_NAME == "ETTh1":
    DEFAULT_SEQ_LEN = 96
    DEFAULT_PRED_LEN = 24
    TRAIN_SIZE = 12 * 30 * 24
    VAL_SIZE = 4 * 30 * 24
    TEST_SIZE = 4 * 30 * 24
else:
    DEFAULT_SEQ_LEN = 96
    DEFAULT_PRED_LEN = 96
    TRAIN_SIZE = 12 * 30 * 24 * 4
    VAL_SIZE = 4 * 30 * 24 * 4
    TEST_SIZE = 4 * 30 * 24 * 4

BATCH_SIZE = 256
EPOCHS = 30
PATIENCE = 6
LR = 1e-3
WEIGHT_DECAY = 1e-4
NUM_BASIS = 16
HIDDEN_DIM = 64
CORE_MODEL_NAMES = ["KAN", "Hybrid KAN", "MLP", "LSTM"]
ABLATION_MODEL_NAMES = ["KAN", "Hybrid KAN"]

In [ ]:
def load_ett_csv(dataset_name):
    url = DATA_URLS[dataset_name]
    local_path = f"{dataset_name}.csv"
    try:
        df = pd.read_csv(local_path)
        print("Локальный файл найден:", local_path)
        return df
    except FileNotFoundError:
        print("Скачиваем:", url)
        response = requests.get(url, timeout=120)
        response.raise_for_status()
        return pd.read_csv(io.StringIO(response.text))

df = load_ett_csv(DATASET_NAME)
df["date"] = pd.to_datetime(df["date"])
df.head()

## 4. Первичный анализ

In [ ]:
print(df.shape)
print(df.isna().sum())
display(df.describe().T)

In [ ]:
plt.figure(figsize=(14, 4))
plt.plot(df["date"][:1000], df[TARGET_COL][:1000])
plt.title(f"{DATASET_NAME}: динамика целевой переменной {TARGET_COL}")
plt.xlabel("Дата")
plt.ylabel(TARGET_COL)
plt.show()

plt.figure(figsize=(8, 5))
sns.heatmap(df[FEATURE_COLS].corr(), annot=True, cmap="YlGnBu", fmt=".2f")
plt.title("Корреляция признаков")
plt.show()

## 5. Стандартные split'ы ETT и подготовка окон

In [ ]:
train_end = TRAIN_SIZE
val_end = TRAIN_SIZE + VAL_SIZE
test_end = TRAIN_SIZE + VAL_SIZE + TEST_SIZE

df = df.iloc[:test_end].copy()

train_df = df.iloc[:train_end].copy()
val_df = df.iloc[train_end:val_end].copy()
test_df = df.iloc[val_end:test_end].copy()

scaler = StandardScaler()
train_scaled = scaler.fit_transform(train_df[FEATURE_COLS])
val_scaled = scaler.transform(val_df[FEATURE_COLS])
test_scaled = scaler.transform(test_df[FEATURE_COLS])

target_idx = FEATURE_COLS.index(TARGET_COL)
target_mean = float(scaler.mean_[target_idx])
target_std = float(scaler.scale_[target_idx])

print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)
print("Target mean:", round(target_mean, 4))
print("Target std:", round(target_std, 4))

In [ ]:
def create_windows(data, seq_len, pred_len, target_index):
    X, y = [], []
    for i in range(len(data) - seq_len - pred_len + 1):
        X.append(data[i:i + seq_len])
        y.append(data[i + seq_len:i + seq_len + pred_len, target_index])
    return np.array(X), np.array(y)

def build_tensors(seq_len=DEFAULT_SEQ_LEN, pred_len=DEFAULT_PRED_LEN):
    X_train, y_train = create_windows(train_scaled, seq_len, pred_len, target_idx)
    X_val, y_val = create_windows(val_scaled, seq_len, pred_len, target_idx)
    X_test, y_test = create_windows(test_scaled, seq_len, pred_len, target_idx)

    return (
        torch.tensor(X_train, dtype=torch.float32),
        torch.tensor(y_train, dtype=torch.float32),
        torch.tensor(X_val, dtype=torch.float32),
        torch.tensor(y_val, dtype=torch.float32),
        torch.tensor(X_test, dtype=torch.float32),
        torch.tensor(y_test, dtype=torch.float32),
    )

X_train, y_train, X_val, y_val, X_test, y_test = build_tensors()
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

## 6. Архитектуры моделей

In [ ]:
class KANLayer(nn.Module):
    def __init__(self, in_dim, out_dim, num_basis=16):
        super().__init__()
        self.knots = nn.Parameter(torch.linspace(-3, 3, num_basis), requires_grad=False)
        self.coeffs = nn.Parameter(torch.randn(in_dim, out_dim, num_basis) * 0.05)

    def forward(self, x):
        x = x.unsqueeze(-1)
        basis = torch.exp(-((x - self.knots) ** 2))
        return torch.einsum("bin,ion->bo", basis, self.coeffs)


class KANForecaster(nn.Module):
    def __init__(self, seq_len, n_features, hidden_dim=64, pred_len=24, num_basis=16):
        super().__init__()
        input_dim = seq_len * n_features
        self.kan1 = KANLayer(input_dim, hidden_dim, num_basis=num_basis)
        self.kan2 = KANLayer(hidden_dim, pred_len, num_basis=num_basis)

    def forward(self, x):
        flat = x.reshape(x.size(0), -1)
        return self.kan2(self.kan1(flat))


class HybridKANForecaster(nn.Module):
    def __init__(self, seq_len, n_features, hidden_dim=64, pred_len=24, num_basis=16):
        super().__init__()
        input_dim = seq_len * n_features
        self.linear = nn.Linear(input_dim, pred_len)
        self.kan1 = KANLayer(input_dim, hidden_dim, num_basis=num_basis)
        self.kan2 = KANLayer(hidden_dim, pred_len, num_basis=num_basis)

    def forward(self, x):
        flat = x.reshape(x.size(0), -1)
        return self.linear(flat) + self.kan2(self.kan1(flat))


class MLPForecaster(nn.Module):
    def __init__(self, seq_len, n_features, hidden_dim=128, pred_len=24):
        super().__init__()
        input_dim = seq_len * n_features
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, pred_len),
        )

    def forward(self, x):
        return self.net(x.reshape(x.size(0), -1))


class LSTMForecaster(nn.Module):
    def __init__(self, n_features, hidden_dim=64, pred_len=24):
        super().__init__()
        self.lstm = nn.LSTM(n_features, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, pred_len)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

## 7. Обучение, baseline-модели и метрики

In [ ]:
def inverse_transform_target(arr):
    return arr * target_std + target_mean


def iterate_minibatches(X, y, batch_size=256, shuffle=True):
    indices = np.arange(len(X))
    if shuffle:
        np.random.shuffle(indices)
    for start in range(0, len(X), batch_size):
        batch_idx = indices[start:start + batch_size]
        yield X[batch_idx], y[batch_idx]


def regression_metrics(y_true_scaled, y_pred_scaled):
    yt = inverse_transform_target(y_true_scaled.reshape(-1))
    yp = inverse_transform_target(y_pred_scaled.reshape(-1))
    return {
        "RMSE": float(np.sqrt(mean_squared_error(yt, yp))),
        "MAE": float(mean_absolute_error(yt, yp)),
        "R2": float(r2_score(yt, yp)),
        "MAPE": float(np.mean(np.abs((yt - yp) / (yt + 1e-8))) * 100),
    }


def evaluate_torch_model(model, X, y):
    model.eval()
    with torch.no_grad():
        preds = model(X.to(DEVICE)).cpu().numpy()
    true = y.cpu().numpy()
    return regression_metrics(true, preds), preds


def predict_last_value_baseline(X):
    X_np = X.cpu().numpy()
    last_values = X_np[:, -1, target_idx]
    return np.repeat(last_values[:, None], X_np.shape[1] * 0 + DEFAULT_PRED_LEN, axis=1)


def predict_last_value_baseline_with_horizon(X, pred_len):
    X_np = X.cpu().numpy()
    last_values = X_np[:, -1, target_idx]
    return np.repeat(last_values[:, None], pred_len, axis=1)


def fit_linear_baseline(X_train, y_train):
    X_train_np = X_train.cpu().numpy().reshape(len(X_train), -1)
    y_train_np = y_train.cpu().numpy()
    model = LinearRegression()
    model.fit(X_train_np, y_train_np)
    return model


def predict_linear_baseline(model, X):
    X_np = X.cpu().numpy().reshape(len(X), -1)
    return model.predict(X_np)


def evaluate_baseline_predictions(y_true, preds):
    true = y_true.cpu().numpy()
    return regression_metrics(true, preds)

In [ ]:
def train_model(model, X_train, y_train, X_val, y_val, epochs=30, lr=1e-3, batch_size=256, patience=6, weight_decay=1e-4):
    model = model.to(DEVICE)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_state = None
    best_val_rmse = math.inf
    best_epoch = 0
    wait = 0
    history = []

    for epoch in range(epochs):
        model.train()
        train_losses = []
        for xb, yb in iterate_minibatches(X_train, y_train, batch_size=batch_size):
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)

            optimizer.zero_grad()
            preds = model(xb)
            loss = criterion(preds, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_losses.append(loss.item())

        val_metrics, _ = evaluate_torch_model(model, X_val, y_val)
        current = {
            "epoch": epoch + 1,
            "train_loss": float(np.mean(train_losses)),
            "val_rmse": val_metrics["RMSE"],
            "val_mae": val_metrics["MAE"],
        }
        history.append(current)

        if current["val_rmse"] < best_val_rmse:
            best_val_rmse = current["val_rmse"]
            best_epoch = epoch + 1
            wait = 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            wait += 1

        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(
                f"Epoch {epoch + 1}/{epochs} | train_loss={current['train_loss']:.4f} | "
                f"val_rmse={current['val_rmse']:.4f}"
            )

        if wait >= patience:
            print(f"Early stopping at epoch {epoch + 1}; best epoch = {best_epoch}")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    return history, best_epoch, best_val_rmse


def run_experiment(
    seq_len=DEFAULT_SEQ_LEN,
    pred_len=DEFAULT_PRED_LEN,
    num_basis=NUM_BASIS,
    epochs=EPOCHS,
    model_names=None,
    include_baselines=True,
):
    if model_names is None:
        model_names = CORE_MODEL_NAMES

    X_train, y_train, X_val, y_val, X_test, y_test = build_tensors(seq_len=seq_len, pred_len=pred_len)

    available_models = {
        "KAN": KANForecaster(seq_len, len(FEATURE_COLS), hidden_dim=HIDDEN_DIM, pred_len=pred_len, num_basis=num_basis),
        "Hybrid KAN": HybridKANForecaster(seq_len, len(FEATURE_COLS), hidden_dim=HIDDEN_DIM, pred_len=pred_len, num_basis=num_basis),
        "MLP": MLPForecaster(seq_len, len(FEATURE_COLS), hidden_dim=128, pred_len=pred_len),
        "LSTM": LSTMForecaster(len(FEATURE_COLS), hidden_dim=HIDDEN_DIM, pred_len=pred_len),
    }

    metrics_map = {}
    preds_map = {}
    history_map = {}
    trained_models = {}
    fit_info = {}

    for name in model_names:
        model = available_models[name]
        print(f"\n{name}")
        history, best_epoch, best_val_rmse = train_model(
            model,
            X_train,
            y_train,
            X_val,
            y_val,
            epochs=epochs,
            lr=LR,
            batch_size=BATCH_SIZE,
            patience=PATIENCE,
            weight_decay=WEIGHT_DECAY,
        )
        metrics, preds = evaluate_torch_model(model, X_test, y_test)
        metrics_map[name] = metrics
        preds_map[name] = preds
        history_map[name] = pd.DataFrame(history)
        trained_models[name] = model
        fit_info[name] = {"best_epoch": best_epoch, "best_val_rmse": best_val_rmse}

    if include_baselines:
        last_value_preds = predict_last_value_baseline_with_horizon(X_test, pred_len)
        metrics_map["Last Value"] = evaluate_baseline_predictions(y_test, last_value_preds)
        preds_map["Last Value"] = last_value_preds
        history_map["Last Value"] = pd.DataFrame()
        fit_info["Last Value"] = {"best_epoch": None, "best_val_rmse": None}

        linear_model = fit_linear_baseline(X_train, y_train)
        linear_preds = predict_linear_baseline(linear_model, X_test)
        metrics_map["Linear Regression"] = evaluate_baseline_predictions(y_test, linear_preds)
        preds_map["Linear Regression"] = linear_preds
        history_map["Linear Regression"] = pd.DataFrame()
        fit_info["Linear Regression"] = {"best_epoch": None, "best_val_rmse": None}

    metrics_df = pd.DataFrame(metrics_map).T.sort_values("RMSE")
    return {
        "metrics": metrics_df,
        "preds": preds_map,
        "history": history_map,
        "models": trained_models,
        "fit_info": fit_info,
        "X_test": X_test,
        "y_test": y_test,
        "seq_len": seq_len,
        "pred_len": pred_len,
        "num_basis": num_basis,
    }

## 8. Основной эксперимент

In [ ]:
results = run_experiment()
results["metrics"]

In [ ]:
metrics_df = results["metrics"].copy()
display(metrics_df)

plt.figure(figsize=(10, 4))
sns.barplot(x=metrics_df.index, y=metrics_df["RMSE"], palette="viridis")
plt.title(f"{DATASET_NAME}: RMSE по моделям")
plt.ylabel("RMSE в исходном масштабе OT")
plt.xlabel("Model")
plt.xticks(rotation=20)
plt.show()

fit_info_df = pd.DataFrame(results["fit_info"]).T
display(fit_info_df)

In [ ]:
plt.figure(figsize=(12, 5))
for model_name, history_df in results["history"].items():
    if not history_df.empty:
        plt.plot(history_df["epoch"], history_df["val_rmse"], label=model_name)
plt.title("Validation RMSE по эпохам")
plt.xlabel("Epoch")
plt.ylabel("Validation RMSE")
plt.legend()
plt.show()

In [ ]:
sample_id = 0
true_unscaled = inverse_transform_target(results["y_test"][sample_id].numpy())

plt.figure(figsize=(12, 4))
plt.plot(true_unscaled, label="True", marker="o", linewidth=2)
for model_name in metrics_df.index:
    pred_unscaled = inverse_transform_target(results["preds"][model_name][sample_id])
    plt.plot(pred_unscaled, label=model_name, alpha=0.8)
plt.title(f"{DATASET_NAME}: прогноз {TARGET_COL} | sample {sample_id}")
plt.xlabel("Forecast horizon step")
plt.ylabel("OT")
plt.legend(ncol=2)
plt.show()

## 9. Интерпретация phi-функций KAN

In [ ]:
def plot_phi_functions(layer, feature_labels=None, out_indices=(0, 1), max_inputs=6):
    x = torch.linspace(-3, 3, 200)
    n_inputs = min(layer.coeffs.shape[0], max_inputs)
    basis = torch.exp(-((x.unsqueeze(-1) - layer.knots.detach().cpu()) ** 2))

    for out_idx in out_indices:
        plt.figure(figsize=(12, 6))
        for in_idx in range(n_inputs):
            coeff = layer.coeffs.detach().cpu()[in_idx, out_idx]
            y = (basis * coeff).sum(dim=1)
            label = feature_labels[in_idx] if feature_labels is not None and in_idx < len(feature_labels) else f"input_{in_idx}"
            plt.plot(x.numpy(), y.numpy(), label=label)
        plt.title(f"phi-функции для выхода {out_idx}")
        plt.xlabel("Input value")
        plt.ylabel("phi(x)")
        plt.legend(ncol=2)
        plt.show()


def build_flat_feature_names(seq_len, feature_cols):
    names = []
    for lag in range(seq_len):
        for col in feature_cols:
            names.append(f"{col}_t-{seq_len - lag}")
    return names

kan_model = results["models"]["KAN"]
flat_feature_names = build_flat_feature_names(results["seq_len"], FEATURE_COLS)
plot_phi_functions(kan_model.kan1, feature_labels=flat_feature_names, out_indices=(0, 1), max_inputs=6)

In [ ]:
def summarize_input_contributions(layer, seq_len, top_k=12):
    coeffs = layer.coeffs.detach().cpu().numpy()
    importance = np.abs(coeffs).sum(axis=(1, 2))
    feature_names = build_flat_feature_names(seq_len, FEATURE_COLS)
    df_imp = pd.DataFrame({
        "feature": feature_names[:len(importance)],
        "importance": importance,
    }).sort_values("importance", ascending=False)
    return df_imp.head(top_k)

top_contrib = summarize_input_contributions(kan_model.kan1, seq_len=results["seq_len"])
display(top_contrib)

plt.figure(figsize=(10, 5))
sns.barplot(data=top_contrib, x="importance", y="feature", palette="mako")
plt.title("Топ вкладов входов в первом KAN-слое")
plt.show()

## 10. Абляция по длине окна, горизонту и числу базисов

Для абляции по умолчанию сравниваются только `KAN` и `Hybrid KAN`, чтобы эксперимент не был слишком тяжёлым.

In [ ]:
def ablation_seq_len(seq_lens=(48, 96, 192), pred_len=DEFAULT_PRED_LEN, epochs=10):
    rows = []
    for seq_len in seq_lens:
        exp = run_experiment(
            seq_len=seq_len,
            pred_len=pred_len,
            num_basis=NUM_BASIS,
            epochs=epochs,
            model_names=ABLATION_MODEL_NAMES,
            include_baselines=False,
        )
        for model_name, row in exp["metrics"].iterrows():
            rows.append({"factor": "seq_len", "value": seq_len, "model": model_name, **row.to_dict()})
    return pd.DataFrame(rows)


def ablation_pred_len(pred_lens=(24, 48, 96), seq_len=DEFAULT_SEQ_LEN, epochs=10):
    rows = []
    for pred_len in pred_lens:
        exp = run_experiment(
            seq_len=seq_len,
            pred_len=pred_len,
            num_basis=NUM_BASIS,
            epochs=epochs,
            model_names=ABLATION_MODEL_NAMES,
            include_baselines=False,
        )
        for model_name, row in exp["metrics"].iterrows():
            rows.append({"factor": "pred_len", "value": pred_len, "model": model_name, **row.to_dict()})
    return pd.DataFrame(rows)


def ablation_num_basis(num_basis_list=(8, 16, 32), seq_len=DEFAULT_SEQ_LEN, pred_len=DEFAULT_PRED_LEN, epochs=10):
    rows = []
    for num_basis in num_basis_list:
        exp = run_experiment(
            seq_len=seq_len,
            pred_len=pred_len,
            num_basis=num_basis,
            epochs=epochs,
            model_names=ABLATION_MODEL_NAMES,
            include_baselines=False,
        )
        for model_name, row in exp["metrics"].iterrows():
            rows.append({"factor": "num_basis", "value": num_basis, "model": model_name, **row.to_dict()})
    return pd.DataFrame(rows)

# Запускайте по одному блоку, когда будете готовы считать абляцию.
# ablation_window_df = ablation_seq_len()
# ablation_horizon_df = ablation_pred_len()
# ablation_basis_df = ablation_num_basis()

In [ ]:
if "ablation_window_df" in globals():
    plot_df = ablation_window_df.copy()
    plt.figure(figsize=(10, 5))
    sns.lineplot(data=plot_df, x="value", y="RMSE", hue="model", marker="o")
    plt.title("Абляция по длине окна")
    plt.xlabel("seq_len")
    plt.ylabel("RMSE")
    plt.show()
else:
    print("Сначала запустите одну из ablation_* функций, затем вернитесь к визуализации.")

## 11. Дипломные выводы, которые можно будет оформить по результатам

1. Проверить, подтверждается ли гипотеза о том, что `Hybrid KAN` устойчивее чистого `KAN`.
2. Сравнить `KAN`-семейство не только с нейросетями, но и с простыми baseline-моделями.
3. Оценить чувствительность `KAN` к длине окна, горизонту и числу базисных функций.
4. Показать, какие лаги и признаки сильнее влияют на прогноз через `phi`-функции.
5. Сопоставить результаты новой главы с уже выполненными экспериментами на недвижимости, кредитных данных и климатических данных.

## 12. Источники для обзора литературы

- Liu et al. `KAN: Kolmogorov-Arnold Networks`, 2024.
- Работы по `ETT` и семейству `Informer / Autoformer` как стандартным benchmark'ам forecasting.
- Современные статьи по `Temporal KAN`, `Hybrid KAN`, `Transformer + KAN` для forecasting.
- В обзоре результатов важно отдельно обсуждать компромисс между качеством прогноза и интерпретируемостью.